In [0]:
%python
from pyspark.sql import SparkSession

host = "61.12.91.141"
port = "54322"
user = "databricks_pennm_user"
password = "ez#5FOr3eFUhG%ps"

jdbc_url = f"jdbc:postgresql://{host}:{port}/ops_compliance_db"

properties = {
    "user": user,
    "password": password,
    "driver": "org.postgresql.Driver"
}

query = """
(
SELECT
    table_schema,
    table_name,
    table_type
FROM information_schema.tables
WHERE table_schema NOT IN ('pg_catalog', 'information_schema')
and table_name not like 'fdw_%'
ORDER BY table_schema, table_name
) t
"""

tables_df = spark.read.jdbc(
    url=jdbc_url,
    table=query,
    properties=properties
)

display(tables_df)

In [0]:
%python
from pyspark.sql import functions as F
from functools import reduce

# Connection details
host = "61.12.91.141"
port = "54322"
user = "databricks_pennm_user"
password = "ez#5FOr3eFUhG%ps"

databases = [
    "ops_compliance_db",
    
    "policy_admin_db"
]

all_dfs = []

for db in databases:
    try:
        jdbc_url = f"jdbc:postgresql://{host}:{port}/{db}"

        query = f"""
        (
        SELECT
            current_database() AS database_name,
            table_schema,
            table_name,
            table_type
        FROM information_schema.tables
        WHERE table_schema NOT IN ('pg_catalog', 'information_schema')
        AND table_type = 'BASE TABLE'
          
        ) t
        """

        df = (
            spark.read
            .format("jdbc")
            .option("url", jdbc_url)
            .option("dbtable", query)
            .option("user", user)
            .option("password", password)
            .option("driver", "org.postgresql.Driver")
            .load()
        )

        all_dfs.append(df)
        print(f"SUCCESS: {db}")

    except Exception as e:
        print(f"FAILED: {db}")
        print(str(e))

if all_dfs:

    tables_df = reduce(lambda x, y: x.unionByName(y), all_dfs)

    print("\n===== TABLE COUNTS BY DATABASE =====")
    display(
        tables_df.groupBy("database_name")
                 .count()
                 .withColumnRenamed("count", "table_count")
                 .orderBy("database_name")
    )

    print("\n===== ALL TABLES =====")
    display(
        tables_df.orderBy(
            "database_name",
            "table_schema",
            "table_name"
        )
    )

else:
    print("No tables found or no database connections succeeded.")

In [0]:
config_df = spark.table("configs.migration_config") \
    .filter("enabled = true") \
    .select("jdbc_url", "source_table", "user", "password", "driver") \
    .collect()

for cfg in config_df:

    jdbc_url = cfg["jdbc_url"]
    source_table = cfg["source_table"]
    user = cfg["user"]
    password = cfg["password"]
    driver = cfg["driver"]

    print("\n" + "="*100)
    print(f"TABLE: {source_table}")
    print("="*100)

    try:
        query = f"(SELECT * FROM {source_table} LIMIT 3) t"

        df = (
            spark.read
            .format("jdbc")
            .option("url", jdbc_url)
            .option("dbtable", query)
            .option("user", user)
            .option("password", password)
            .option("driver", driver)
            .load()
        )

        display(df)

    except Exception as e:
        print(f"ERROR reading {source_table}: {str(e)}")

In [0]:
%sql
create schema bronze.ops_compliance_db;
create schema bronze.policy_admin_db;

In [0]:
from pyspark.sql import Row

tables = [
    "bronze.ops_compliance_db.premium_payments"
,"bronze.ops_compliance_db.audit_log"
,"bronze.ops_compliance_db.billing_accounts"
,"bronze.ops_compliance_db.documents"
,"bronze.ops_compliance_db.compliance_events"
,"bronze.ops_compliance_db.claims"
,"bronze.ops_compliance_db.policy_dividends"
,"bronze.ops_compliance_db.underwriting_assessments"
,"bronze.policy_admin_db.applications"
,"bronze.policy_admin_db.policy_status_history"
,"bronze.policy_admin_db.medical_records"
,"bronze.policy_admin_db.addresses"
,"bronze.policy_admin_db.agents"
,"bronze.policy_admin_db.beneficiaries"
,"bronze.policy_admin_db.policies"
,"bronze.policy_admin_db.policy_riders"
,"bronze.policy_admin_db.customers"
]

rows = []

for table in tables:
    for col in spark.table(table).schema.fields:
        rows.append(
            Row(
                table_name=table,
                column_name=col.name,
                data_type=str(col.dataType),
                nullable=col.nullable
            )
        )

display(spark.createDataFrame(rows))

In [0]:
-- =========================================================
-- GOLD LAYER DATA MART DOCUMENTATION
-- SCHEMA: gold.gold_tables
-- =========================================================


-- =========================================================
-- 1. CUSTOMER 360 TABLE
-- =========================================================
-- PURPOSE:
-- Provides a unified 360-degree view of each customer by combining:
-- - Customer personal details
-- - Applications submitted by the customer
-- - Policies issued to the customer
-- - Assigned agent details

-- BUSINESS VALUE:
-- This is the most important analytical table for customer analytics.

-- ANSWERS BUSINESS QUESTIONS:
-- * Who are our customers and what are their demographics?
-- * Which customers have active policies?
-- * What products have they applied for?
-- * Which agent manages which customer?
-- * What is the customer’s total insurance footprint?

-- USE CASES:
-- Customer segmentation, cross-sell analysis, CRM dashboards, churn analysis


-- =========================================================
-- 2. POLICY PERFORMANCE MART
-- =========================================================
-- PURPOSE:
-- Aggregates policy-level financials and lifecycle metrics including riders.

-- BUSINESS VALUE:
-- Helps evaluate profitability and longevity of insurance policies.

-- ANSWERS BUSINESS QUESTIONS:
-- * How long do policies stay active?
-- * What is the total rider value per policy?
-- * Which policies are high value vs low value?
-- * What is the current age of each policy?

-- USE CASES:
-- Revenue analysis, product performance tracking, policy profitability reports


-- =========================================================
-- 3. CLAIMS ANALYSIS MART
-- =========================================================
-- PURPOSE:
-- Provides detailed view of insurance claims with status classification.

-- BUSINESS VALUE:
-- Enables claims management and fraud detection insights.

-- ANSWERS BUSINESS QUESTIONS:
-- * How many claims are approved, rejected, or pending?
-- * What is the outstanding claim liability?
-- * Which policies generate the highest claims?
-- * What is the claim settlement rate?

-- USE CASES:
-- Claims monitoring dashboard, fraud analytics, financial provisioning


-- =========================================================
-- 4. PREMIUM PAYMENT PERFORMANCE
-- =========================================================
-- PURPOSE:
-- Tracks all premium payments with status and delay analysis.

-- BUSINESS VALUE:
-- Helps monitor cash flow and customer payment behavior.

-- ANSWERS BUSINESS QUESTIONS:
-- * Which payments are delayed or missed?
-- * What is the payment success rate?
-- * Which billing accounts are high risk?
-- * How many payments are completed vs failed?

-- USE CASES:
-- Revenue tracking, payment delinquency reports, financial reconciliation


-- =========================================================
-- 5. UNDERWRITING RISK VIEW
-- =========================================================
-- PURPOSE:
-- Normalizes underwriting assessments into risk categories.

-- BUSINESS VALUE:
-- Supports risk-based pricing and policy approval decisions.

-- ANSWERS BUSINESS QUESTIONS:
-- * What is the health risk of applicants?
-- * How many applicants are high-risk smokers?
-- * What is the BMI-based risk distribution?
-- * How many applications are approved or rejected?

-- USE CASES:
-- Risk scoring models, underwriting dashboards, pricing strategy


-- =========================================================
-- 6. POLICY LIFECYCLE HISTORY
-- =========================================================
-- PURPOSE:
-- Tracks all policy status changes over time.

-- BUSINESS VALUE:
-- Enables full lifecycle tracking of policies.

-- ANSWERS BUSINESS QUESTIONS:
-- * How often do policies lapse or change status?
-- * What are the common reasons for status change?
-- * When did a policy move from active to lapsed?

-- USE CASES:
-- Policy churn analysis, regulatory reporting, lifecycle tracking


-- =========================================================
-- 7. AGENT PERFORMANCE TABLE
-- =========================================================
-- PURPOSE:
-- Measures performance of insurance agents across applications and policies.

-- BUSINESS VALUE:
-- Supports sales performance evaluation and incentive programs.

-- ANSWERS BUSINESS QUESTIONS:
-- * Which agents generate the most policies?
-- * Who brings the highest premium revenue?
-- * Which agency is most productive?
-- * How many applications convert to policies per agent?

-- USE CASES:
-- Sales dashboards, commission calculation, agent ranking


-- =========================================================
-- 8. BENEFICIARY DISTRIBUTION
-- =========================================================
-- PURPOSE:
-- Shows how policy benefits are distributed among beneficiaries.

-- BUSINESS VALUE:
-- Supports claims settlement and legal compliance.

-- ANSWERS BUSINESS QUESTIONS:
-- * Who are the beneficiaries of each policy?
-- * What percentage does each beneficiary receive?
-- * Are there minor beneficiaries involved?
-- * Which policies have complex beneficiary structures?

-- USE CASES:
-- Claims payout planning, legal verification, family coverage analysis


-- =========================================================
-- 9. COMPLIANCE AUDIT VIEW
-- =========================================================
-- PURPOSE:
-- Tracks all audit logs for compliance and governance.

-- BUSINESS VALUE:
-- Ensures regulatory compliance and traceability of system changes.

-- ANSWERS BUSINESS QUESTIONS:
-- * Who made changes to critical tables?
-- * What operations were performed (INSERT/UPDATE/DELETE)?
-- * From which IP or system was the change made?
-- * When was the last modification done?

-- USE CASES:
-- Audit reporting, compliance checks, security monitoring


-- =========================================================
-- 10. COMPLIANCE EVENTS SUMMARY
-- =========================================================
-- PURPOSE:
-- Provides summary view of regulatory/compliance events.

-- BUSINESS VALUE:
-- Helps monitor risk and regulatory violations across systems.

-- ANSWERS BUSINESS QUESTIONS:
-- * How many compliance issues are open vs resolved?
-- * What severity levels are most common?
-- * Which entities have frequent violations?
-- * What is the resolution rate of compliance events?

-- USE CASES:
-- Regulatory reporting, risk monitoring, governance dashboards


-- =========================================================
-- GOLD LAYER TABLES (BUSINESS DATA MARTS)
-- SCHEMA: gold.gold_tables
-- PURPOSE: Business-ready analytical layer for reporting,
--          dashboards, KPI tracking, and decision making
-- =========================================================


-- =========================================================
-- 1. CUSTOMER 360 VIEW
-- =========================================================
-- BUSINESS PURPOSE:
-- Provides a unified 360-degree view of each customer by combining
-- personal details, applications, policies, and assigned agents.

-- BUSINESS VALUE:
-- Enables customer segmentation, cross-sell/upsell strategies,
-- and customer lifecycle tracking.

-- QUESTIONS IT ANSWERS:
-- - What policies does a customer hold?
-- - What is the customer's income and risk profile?
-- - Which agent manages this customer?
-- - Which customers are high-value or underinsured?

CREATE Table gold.gold_tables.customer_360 AS
SELECT
    c.customer_id,
    c.first_name,
    c.last_name,
    c.gender,
    c.date_of_birth,
    c.annual_income,
    c.occupation,
    c.email,
    c.phone,
    c.is_active,

    a.application_id,
    a.application_type,
    a.application_date,
    a.coverage_amount,
    a.status AS application_status,
    a.underwriting_status,

    p.policy_id,
    p.policy_number,
    p.policy_type,
    p.status AS policy_status,
    p.coverage_amount AS policy_coverage,
    p.premium_amount,
    p.annual_premium,
    p.start_date,
    p.end_date,

    ag.agent_id,
    ag.agent_name,
    ag.agency_name

FROM silver.policy_admin_db.customers c
LEFT JOIN silver.policy_admin_db.applications a
    ON c.customer_id = a.customer_id
LEFT JOIN silver.policy_admin_db.policies p
    ON a.application_id = p.application_id
LEFT JOIN silver.policy_admin_db.agents ag
    ON a.agent_id = ag.agent_id;


-- =========================================================
-- 2. POLICY PERFORMANCE MART
-- =========================================================
-- BUSINESS PURPOSE:
-- Tracks policy lifecycle performance including duration,
-- coverage, premium, and rider contributions.

-- BUSINESS VALUE:
-- Helps actuarial teams and finance understand revenue contribution
-- and policy profitability.

-- QUESTIONS IT ANSWERS:
-- - How long are policies active?
-- - What is total rider revenue per policy?
-- - Which policies generate highest premium?
-- - What is policy lifecycle duration?

CREATE Table gold.gold_tables.policy_performance AS
SELECT
    p.policy_id,
    p.policy_number,
    p.policy_type,
    p.status,
    p.coverage_amount,
    p.premium_amount,
    p.annual_premium,
    p.billing_frequency,
    p.start_date,
    p.end_date,

    DATEDIFF(p.end_date, p.start_date) AS policy_duration_days,
    DATEDIFF(current_date(), p.start_date) AS policy_age_days,

    SUM(pr.rider_amount) AS total_rider_amount

FROM silver.policy_admin_db.policies p
LEFT JOIN silver.policy_admin_db.policy_riders pr
    ON p.policy_id = pr.policy_id
GROUP BY
    p.policy_id,
    p.policy_number,
    p.policy_type,
    p.status,
    p.coverage_amount,
    p.premium_amount,
    p.annual_premium,
    p.billing_frequency,
    p.start_date,
    p.end_date;


-- =========================================================
-- 3. CLAIMS ANALYTICS MART
-- =========================================================
-- BUSINESS PURPOSE:
-- Provides visibility into insurance claims lifecycle and payout behavior.

-- BUSINESS VALUE:
-- Supports fraud detection, claims settlement efficiency, and risk monitoring.

-- QUESTIONS IT ANSWERS:
-- - How many claims are approved vs rejected?
-- - What is total claim liability?
-- - Which policies generate highest claims?
-- - Are claims being settled fully or partially?

CREATE Table gold.gold_tables.claims_analysis AS
SELECT
    cl.claim_id,
    cl.policy_id,
    cl.customer_id,
    cl.claim_type,
    cl.claim_amount,
    cl.approved_amount,
    cl.paid_amount,
    cl.status,
    cl.claim_date,
    cl.incident_date,
    cl.denial_reason,
    cl.investigator,

    CASE
        WHEN cl.status = 'APPROVED' THEN 'SETTLED'
        WHEN cl.status = 'REJECTED' THEN 'DENIED'
        ELSE 'PENDING'
    END AS claim_status_category,

    (cl.approved_amount - cl.paid_amount) AS outstanding_amount

FROM silver.ops_compliance_db.claims cl;


-- =========================================================
-- 4. PREMIUM PAYMENT PERFORMANCE
-- =========================================================
-- BUSINESS PURPOSE:
-- Tracks customer premium payment behavior and delays.

-- BUSINESS VALUE:
-- Helps revenue assurance, churn prediction, and cashflow forecasting.

-- QUESTIONS IT ANSWERS:
-- - Who is paying late premiums?
-- - What percentage of payments fail?
-- - Which customers are risky in payment behavior?

CREATE Table gold.gold_tables.premium_payment_performance AS
SELECT
    pp.payment_id,
    pp.billing_account_id,
    pp.amount,
    pp.payment_status,
    pp.payment_method,
    pp.due_date,
    pp.paid_date,
    pp.days_late,

    CASE
        WHEN pp.days_late > 0 THEN TRUE ELSE FALSE
    END AS is_late_payment,

    CASE
        WHEN UPPER(pp.payment_status) = 'SUCCESS' THEN 'COMPLETED'
        WHEN UPPER(pp.payment_status) = 'FAILED' THEN 'FAILED'
        ELSE 'PENDING'
    END AS payment_category

FROM silver.ops_compliance_db.premium_payments pp;


-- =========================================================
-- 5. UNDERWRITING RISK VIEW
-- =========================================================
-- BUSINESS PURPOSE:
-- Evaluates risk profile of applicants using BMI and smoking behavior.

-- BUSINESS VALUE:
-- Supports underwriting decisions, pricing, and risk classification.

-- QUESTIONS IT ANSWERS:
-- - Who are high-risk applicants?
-- - What is BMI distribution of customers?
-- - How does smoking impact risk category?

CREATE Table gold.gold_tables.underwriting_risk AS
SELECT
    ua.assessment_id,
    ua.application_id,
    ua.bmi,
    ua.smoker,
    ua.risk_class,
    ua.table_rating,
    ua.decision,

    CASE
        WHEN ua.bmi >= 30 THEN 'OBESE'
        WHEN ua.bmi >= 25 THEN 'OVERWEIGHT'
        ELSE 'NORMAL'
    END AS bmi_category,

    CASE
        WHEN ua.smoker = TRUE THEN 'HIGH_RISK'
        ELSE 'LOW_RISK'
    END AS smoking_risk_flag

FROM silver.ops_compliance_db.underwriting_assessments ua;


-- =========================================================
-- 6. POLICY LIFECYCLE HISTORY
-- =========================================================
-- BUSINESS PURPOSE:
-- Tracks historical policy status changes over time.

-- BUSINESS VALUE:
-- Helps churn analysis, lapse prediction, and policy retention insights.

-- QUESTIONS IT ANSWERS:
-- - How often do policies lapse?
-- - What causes policy status changes?
-- - What is policy renewal behavior?

CREATE Table gold.gold_tables.policy_lifecycle AS
SELECT
    ph.history_id,
    ph.policy_id,
    ph.previous_status,
    ph.new_status,
    ph.change_reason,
    ph.changed_at,

    CASE
        WHEN ph.new_status = 'LAPSED' THEN 1 ELSE 0
    END AS is_lapse_event

FROM silver.policy_admin_db.policy_status_history ph;


-- =========================================================
-- 7. AGENT PERFORMANCE
-- =========================================================
-- BUSINESS PURPOSE:
-- Measures productivity and sales performance of insurance agents.

-- BUSINESS VALUE:
-- Used for commissions, incentive programs, and sales optimization.

-- QUESTIONS IT ANSWERS:
-- - Which agent generates most revenue?
-- - How many policies per agent?
-- - Which agency performs best?

CREATE Table gold.gold_tables.agent_performance AS
SELECT
    ag.agent_id,
    ag.agent_name,
    ag.agency_name,

    COUNT(DISTINCT a.application_id) AS total_applications,
    COUNT(DISTINCT p.policy_id) AS total_policies,
    SUM(p.premium_amount) AS total_premium_generated

FROM silver.policy_admin_db.agents ag
LEFT JOIN silver.policy_admin_db.applications a
    ON ag.agent_id = a.agent_id
LEFT JOIN silver.policy_admin_db.policies p
    ON a.application_id = p.application_id

GROUP BY
    ag.agent_id,
    ag.agent_name,
    ag.agency_name;


-- =========================================================
-- 8. BENEFICIARY DISTRIBUTION
-- =========================================================
-- BUSINESS PURPOSE:
-- Shows how policy benefits are distributed among beneficiaries.

-- BUSINESS VALUE:
-- Used in claims settlement validation and legal verification.

-- QUESTIONS IT ANSWERS:
-- - Who are beneficiaries of a policy?
-- - What percentage allocation exists?
-- - Are minors involved in policy claims?

CREATE Table gold.gold_tables.beneficiary_distribution AS
SELECT
    b.beneficiary_id,
    b.policy_id,
    b.full_name,
    b.relationship,
    b.allocation_percent,
    b.is_minor,

    p.policy_number,
    p.coverage_amount

FROM silver.policy_admin_db.beneficiaries b
LEFT JOIN silver.policy_admin_db.policies p
    ON b.policy_id = p.policy_id;


-- =========================================================
-- 9. AUDIT COMPLIANCE VIEW
-- =========================================================
-- BUSINESS PURPOSE:
-- Tracks all system-level data changes and user actions.

-- BUSINESS VALUE:
-- Ensures regulatory compliance and fraud monitoring.

-- QUESTIONS IT ANSWERS:
-- - Who changed what data and when?
-- - Which tables are frequently modified?
-- - Is there suspicious activity in the system?

CREATE Table gold.gold_tables.compliance_audit AS
SELECT
    al.audit_id,
    al.table_name,
    al.operation,
    al.changed_by,
    al.changed_at,
    al.client_ip,
    al.application_name

FROM silver.ops_compliance_db.audit_log al;


-- =========================================================
-- 10. COMPLIANCE EVENTS SUMMARY
-- =========================================================
-- BUSINESS PURPOSE:
-- Tracks regulatory compliance issues and resolution status.

-- BUSINESS VALUE:
-- Helps compliance teams monitor unresolved issues and severity.

-- QUESTIONS IT ANSWERS:
-- - What compliance events are open vs closed?
-- - Which entities have highest violations?
-- - How fast are issues resolved?

CREATE Table gold.gold_tables.compliance_events_summary AS
SELECT
    ce.event_id,
    ce.entity_type,
    ce.entity_id,
    ce.event_type,
    ce.severity,
    ce.event_date,
    ce.is_resolved,

    CASE
        WHEN ce.is_resolved = TRUE THEN 'CLOSED'
        ELSE 'OPEN'
    END AS event_status

FROM silver.ops_compliance_db.compliance_events ce;

In [0]:
from pyspark.sql import Row

tables = [
    "bronze.ops_compliance_db.premium_payments"
,"bronze.ops_compliance_db.audit_log"
,"bronze.ops_compliance_db.billing_accounts"
,"bronze.ops_compliance_db.documents"
,"bronze.ops_compliance_db.compliance_events"
,"bronze.ops_compliance_db.claims"
,"bronze.ops_compliance_db.policy_dividends"
,"bronze.ops_compliance_db.underwriting_assessments"
,"bronze.policy_admin_db.applications"
,"bronze.policy_admin_db.policy_status_history"
,"bronze.policy_admin_db.medical_records"
,"bronze.policy_admin_db.addresses"
,"bronze.policy_admin_db.agents"
,"bronze.policy_admin_db.beneficiaries"
,"bronze.policy_admin_db.policies"
,"bronze.policy_admin_db.policy_riders"
,"bronze.policy_admin_db.customers"
]

rows = []

for table in tables:
    for col in spark.table(table).schema.fields:
        rows.append(
            Row(
                table_name=table,
                column_name=col.name,
            )
        )

display(spark.createDataFrame(rows))

In [0]:
%sql
Select * from bronze.ops_compliance_db.premium_payments where billing_account_id='06cfab07-9944-4dce-b0a0-303cc6a2aa4c'